In [1]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/notebooks/farahlicir/online-shoppers-intention-data-splitting-prepro/train.csv")
val_df = pd.read_csv("/kaggle/input/notebooks/farahlicir/online-shoppers-intention-data-splitting-prepro/validation.csv")
test_df = pd.read_csv("/kaggle/input/notebooks/farahlicir/online-shoppers-intention-data-splitting-prepro/test.csv")

In [2]:
X_train = train_df.drop(columns="Revenue")
y_train = train_df["Revenue"]

X_val = val_df.drop(columns="Revenue")
y_val = val_df["Revenue"]

X_test = test_df.drop(columns="Revenue")
y_test = test_df["Revenue"]

In [3]:
import joblib

preprocessor = joblib.load("/kaggle/input/notebooks/farahlicir/online-shoppers-intention-data-splitting-prepro/preprocessor.joblib")

In [4]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [5]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [6]:
y_val_pred = logistic_model.predict(X_val_processed)

In [7]:
y_val_proba = logistic_model.predict_proba(X_val_processed)[:, 1]

In [8]:
from sklearn.metrics import classification_report

print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

       False       0.90      0.97      0.93      1648
        True       0.70      0.39      0.50       305

    accuracy                           0.88      1953
   macro avg       0.80      0.68      0.71      1953
weighted avg       0.86      0.88      0.86      1953



In [9]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_val, y_val_proba))
print("PR-AUC :", average_precision_score(y_val, y_val_proba))

ROC-AUC: 0.876442384211364
PR-AUC : 0.5911163171933895


In [10]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_proba
)

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_threshold = (y_val_proba >= threshold).astype(int)

    p = precision_score(y_val, y_pred_threshold, zero_division=0)
    r = recall_score(y_val, y_pred_threshold, zero_division=0)
    f1 = f1_score(y_val, y_pred_threshold, zero_division=0)

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={p:.3f} | "
        f"Recall={r:.3f} | "
        f"F1={f1:.3f}"
    )

Threshold=0.2 | Precision=0.537 | Recall=0.672 | F1=0.597
Threshold=0.3 | Precision=0.645 | Recall=0.541 | F1=0.588
Threshold=0.4 | Precision=0.676 | Recall=0.452 | F1=0.542
Threshold=0.5 | Precision=0.698 | Recall=0.387 | F1=0.498
Threshold=0.6 | Precision=0.685 | Recall=0.321 | F1=0.438
Threshold=0.7 | Precision=0.683 | Recall=0.275 | F1=0.393


In [12]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.96, 0.01)

results = []

for threshold in thresholds:
    y_pred_threshold = (y_val_proba >= threshold).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, y_pred_threshold, zero_division=0
        ),
        "recall": recall_score(
            y_val, y_pred_threshold, zero_division=0
        ),
        "f1": f1_score(
            y_val, y_pred_threshold, zero_division=0
        )
    })

threshold_results = pd.DataFrame(results)

threshold_results.loc[
    threshold_results["f1"].idxmax()
]

threshold    0.230000
precision    0.589506
recall       0.626230
f1           0.607313
Name: 18, dtype: float64

In [13]:
y_test_proba = logistic_model.predict_proba(X_test_processed)[:, 1]

In [14]:
final_threshold = 0.23

y_test_pred = (y_test_proba >= final_threshold).astype(int)

In [15]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("PR-AUC:", average_precision_score(y_test, y_test_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Precision: 0.6064814814814815
Recall: 0.6858638743455497
F1: 0.6437346437346437
ROC-AUC: 0.9011846852917469
PR-AUC: 0.655621059760282

Confusion Matrix:
[[1889  170]
 [ 120  262]]
